# RAGAS — Evaluating RAG Pipelines

RAGAS (Retrieval Augmented Generation Assessment) is a framework used to **evaluate the quality of a RAG (Retrieval-Augmented Generation) pipeline** without needing a human to manually check every answer.

A RAG system has two moving parts that can each go wrong:
1. **Retriever** — did it fetch the right chunks of context from the knowledge base?
2. **Generator (LLM)** — did it use that context correctly to produce a good answer?

RAGAS gives you separate metrics to diagnose *which* part is failing.

## Ground Truth

**Ground Truth** is the ideal, correct answer to a question — written or verified by a human (or a trusted reference source) — that you treat as the "gold standard."

It is used as a reference point to check how close your RAG system's generated answer actually is to the correct answer.

Example:
- **Question:** "What is the capital of France?"
- **Ground Truth:** "Paris"
- **Model's Answer:** "The capital of France is Paris."

Even though the wording differs, a good evaluation metric should recognize these as matching in meaning.

> Note: Ground truth is only required for a couple of RAGAS metrics (like *context recall* and *answer correctness*). Metrics like *faithfulness* and *answer relevancy* don't need it — they work by comparing the answer against the retrieved context and the question instead.

## The Dataset

To run a RAGAS evaluation, you build a small dataset where each row typically contains:

| Column | Meaning |
|---|---|
| `question` | The user query sent to the RAG pipeline |
| `answer` | The answer generated by your RAG system |
| `contexts` | The list of text chunks the retriever pulled from the knowledge base |
| `ground_truth` | The correct reference answer (needed for some metrics) |

RAGAS scores every row on the metrics you choose, then aggregates them into an overall score per metric.

In [ ]:
# Install RAGAS and dependencies
!pip install ragas datasets langchain-openai -q

In [ ]:
from datasets import Dataset

# Example evaluation dataset (in practice this comes from running your RAG pipeline)
data = {
    "question": [
        "What is the capital of France?",
        "Who wrote the play Romeo and Juliet?"
    ],
    "answer": [
        "The capital of France is Paris.",
        "Romeo and Juliet was written by William Shakespeare."
    ],
    "contexts": [
        ["Paris is the capital and most populous city of France."],
        ["Romeo and Juliet is a tragedy written by William Shakespeare early in his career."]
    ],
    "ground_truth": [
        "Paris",
        "William Shakespeare"
    ]
}

eval_dataset = Dataset.from_dict(data)
eval_dataset

## The 4 Core RAGAS Metrics

You wrote down **"metrices — 4, faithfulness, relevance"** — these are two of the four classic RAGAS metrics. Here's all four, explained:

### 1. Faithfulness
Measures whether the generated **answer is factually grounded in the retrieved context** — i.e., does the model make things up (hallucinate) or does every claim in the answer trace back to the context?
- Score range: 0 to 1 (1 = fully grounded, no hallucination)
- How it works: RAGAS breaks the answer into individual claims/statements, then checks whether each one can be inferred from the given context.

### 2. Answer Relevancy
Measures whether the generated **answer actually addresses the question asked** — not off-topic, not overly verbose, not evasive.
- How it works: RAGAS generates several artificial questions *from* the answer, then measures how similar those questions are (via embeddings) to the original question. A high similarity means the answer was on-point.

### 3. Context Precision
Measures whether the **retrieved context chunks are actually relevant** to the question — i.e., did the retriever fetch mostly useful chunks, or a lot of noise?
- Checks the ranking too: relevant chunks should ideally appear near the top of the retrieved list.

### 4. Context Recall
Measures whether the retriever **fetched all the information needed** to answer the question completely, compared against the ground truth.
- This is the metric that *requires* `ground_truth`, since it checks coverage against the known correct answer.

**Quick mental model:**
| Metric | Checks | Needs Ground Truth? |
|---|---|---|
| Faithfulness | Answer vs Context (no hallucination) | No |
| Answer Relevancy | Answer vs Question (on-topic) | No |
| Context Precision | Retrieved chunks vs Question (low noise) | No |
| Context Recall | Retrieved chunks vs Ground Truth (full coverage) | Yes |

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

result = evaluate(
    eval_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
)

result

In [ ]:
# Convert to a pandas DataFrame for a per-question breakdown
df = result.to_pandas()
df

## How to Read the Results

- **Low faithfulness** → your LLM is hallucinating; try tightening the prompt to say "only answer from the given context."
- **Low answer relevancy** → the model is answering a different question than asked, or padding with irrelevant detail.
- **Low context precision** → your retriever is pulling in too much irrelevant text; consider a better embedding model, reranker, or smaller chunk size.
- **Low context recall** → your retriever is missing key information; consider increasing `top_k`, improving chunking, or expanding your knowledge base.

Running these four together lets you pinpoint *whether the retriever or the generator* is the weak link in your RAG pipeline — instead of just eyeballing whether answers "look right."